# Before your agent pays: run a conformity check and get certified

Shopping agents make a predictable kind of mistake: the wrong
variant, a total over the buyer's limit, a fit fact that lived only
in the product prose. The failure is rarely intelligence — it is
that nothing in the flow *checks the built cart against what the
buyer actually authorized* before the money moves.

This cookbook builds that check into the flow as a normal step —
build, check, fix or ask, then pay — using
[ProveCart](https://provecart.com), a neutral verification layer
for agent commerce:

1. The buyer's instructions are captured as a signed **mandate**.
2. Before payment, the built cart is sent to the conformity check
   (Agent Correction Format, `com.provecart.conformity`).
3. The signed verdict routes the flow:
   - **MATCH** → proceed to payment.
   - **BLOCK** with a correction payload → apply only the named
     repairs that stay inside the buyer's authorization, resubmit,
     and let the verdict decide again.
   - **REVIEW** → stop and ask the buyer. Never guess past it.
4. The last section covers **certification**: the same habits,
   proven on a graded exam an agent takes by itself.

ProveCart never holds money; the verdict gates the capture in the
merchant's own checkout.


## Setup

By default the notebook runs **offline**: a local mock stands in for the ProveCart API, so every cell runs top-to-bottom with no credentials and nothing is created. To run the same cells against the live services, set `LIVE_MODE = True` in the setup cell below — then you need an OpenAI API key (`OPENAI_API_KEY`) and a ProveCart integrator key (`PROVECART_API_KEY`) in the environment. The conformity client below uses only the Python standard library.

In [ ]:
%pip install -q openai-agents

import os

# Offline by default. With LIVE_MODE = False (the default), every
# ProveCart call below is answered by a local mock: nothing is
# created, no credentials are needed, and the verdict-routing
# logic can be exercised top-to-bottom. Set LIVE_MODE = True and
# provide both API keys to run the same cells against the live
# services.
LIVE_MODE = False

if LIVE_MODE:
    assert os.environ.get("OPENAI_API_KEY"), "set OPENAI_API_KEY"
    assert os.environ.get("PROVECART_API_KEY"), "set PROVECART_API_KEY"

## The conformity client

One small client for the two calls that matter — register the mandate, verify the cart — plus the published consumer rules that route a verdict.

In [ ]:
import json
import urllib.error
import urllib.request

PROVECART_BASE = "https://api.provecart.com"

# The published fix vocabulary of the Agent Correction Format
# (ACF v0.2, com.provecart.conformity). A BLOCK may only be
# auto-fixed when EVERY correction uses a known action AND the
# repair brings the cart INTO the buyer's existing authorization.
KNOWN_ACTIONS = {
    "remove_item", "replace_item", "change_variant", "reduce_quantity",
    "reduce_total", "change_currency", "correct_traveler_name",
    "change_airport", "change_travel_date", "change_departure_time",
    "add_itinerary", "rebuild_itinerary", "change_cabin",
    "change_fare_terms", "add_hotel_stay", "change_room",
    "change_provider", "change_location", "change_date", "change_time",
    "change_party_size", "correct_reservation_name", "reduce_deposit",
    "change_cancellation_terms",
}


MANDATES_BY_ID = {}


def mandate_for_id(mandate_id):
    """The signed mandate this consumer holds under mandate_id,
    or None. Routing must use the mandate the check was run
    against: a correction that is inside one buyer's mandate
    can be outside another's."""
    return MANDATES_BY_ID.get(mandate_id)


def _mock_post(path, body):
    """Offline stand-in for the ProveCart API, used whenever
    LIVE_MODE is False. Replays the verdicts this walkthrough is
    built around, decided from the request body alone; nothing is
    created or stored."""
    if path == "/mandates":
        return 200, {"id": "mandate_offline_demo"}
    if path.startswith("/agents/signup"):
        return 200, {"agent_id": "agent_offline_demo",
                     "api_key": "offline-demo-key"}
    if path == "/verify":
        items = body.get("items") or []
        variant = (items[0].get("variant") if items else None) or ""
        if body.get("previous_record_id"):
            # A resubmission linked to the check it corrects.
            return 200, {"verdict": "MATCH",
                         "record_id": "rec_offline_2"}
        if variant == "standard frame":
            return 200, {
                "verdict": "BLOCK", "record_id": "rec_offline_1",
                "max_attempts": 3,
                "corrections": [{
                    "action": "change_variant",
                    "field": "items[0].variant",
                    "expected": "step-through frame",
                    "actual": "standard frame"}]}
        # The instruction does not settle the choice: ask the buyer.
        return 200, {"verdict": "REVIEW",
                     "record_id": "rec_offline_3"}
    raise RuntimeError(f"no offline response for {path}")


def pc_post(path, body, api_key):
    if not LIVE_MODE:
        return _mock_post(path, body)
    req = urllib.request.Request(
        PROVECART_BASE + path,
        data=json.dumps(body).encode(),
        method="POST",
        headers={"Content-Type": "application/json",
                 "X-API-Key": api_key})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            return resp.status, json.loads(resp.read())
    except urllib.error.HTTPError as e:
        try:
            return e.code, json.loads(e.read() or b"{}")
        except Exception:
            return e.code, {}


def create_mandate(api_key, **fields):
    """Register the buyer's signed instructions; returns the mandate."""
    status, out = pc_post("/mandates", fields, api_key)
    if status != 200:
        raise RuntimeError(f"mandate creation failed: {status} {out}")
    # Keep the buyer-signed fields with the returned id: the
    # consumer holds its own copy of the mandate, and
    # route_verdict checks corrections against it.
    merged = {**fields, **out}
    MANDATES_BY_ID[merged["id"]] = merged
    return merged


def verify_cart(api_key, mandate_id, merchant, items, total_price,
                currency="USD", previous_record_id=None):
    """Check one built cart against the mandate."""
    body = {"mandate_id": mandate_id, "merchant": merchant,
            "items": items, "total_price": total_price,
            "currency": currency, "cart_changed": bool(previous_record_id)}
    if previous_record_id:
        body["previous_record_id"] = previous_record_id
    status, out = pc_post("/verify", body, api_key)
    if status != 200:
        raise RuntimeError(f"verify failed: {status} {out}")
    return out


def _correction_authorized(correction, mandate):
    """True only when this correction's proposed value can be
    established from the mandate the buyer actually signed.
    A known action name is NOT enough: change_currency to EUR is
    not a fix when the mandate authorizes USD. Anything that
    cannot be established here escalates to the buyer."""
    if mandate is None:
        return False
    action = correction.get("action")
    field = correction.get("field") or ""
    expected = correction.get("expected")
    if action == "remove_item":
        # Removing only shrinks the cart; it cannot widen
        # what the buyer authorized.
        return True
    if expected is None:
        return False
    if action == "change_currency" or field == "currency":
        return mandate.get("currency") is not None \
            and expected == mandate["currency"]
    if action == "reduce_total" or field in ("total_price", "total"):
        try:
            return mandate.get("max_total_price") is not None \
                and float(expected) <= float(mandate["max_total_price"])
        except (TypeError, ValueError):
            return False
    if action == "reduce_quantity" or field.endswith("quantity"):
        try:
            return mandate.get("max_quantity") is not None \
                and float(expected) <= float(mandate["max_quantity"])
        except (TypeError, ValueError):
            return False
    if field.startswith("items") or action in ("change_variant",
                                               "replace_item"):
        # The corrected value must be one the buyer's own words
        # asked for: it appears in the mandate's description or
        # keywords (e.g. "step-through frame").
        haystack = ((mandate.get("item_description") or "") + " "
                    + " ".join(mandate.get("item_keywords")
                               or [])).lower()
        return str(expected).lower() in haystack
    # Any other action (travel, reservations, ...): this consumer
    # holds no mandate field that establishes the value, so it
    # cannot auto-fix. Extend this mapping for your mandate type;
    # until then the correction escalates, per the ACF rules.
    return False


def route_verdict(response, mandate=None):
    """ACF consumer rules. Returns (decision, detail) where decision
    is 'proceed', 'fix' or 'escalate'. A BLOCK is auto-fixed only
    when EVERY correction uses a known action AND its proposed
    value is established by the signed mandate."""
    verdict = response.get("verdict")
    if verdict == "MATCH":
        return "proceed", {"record_id": response.get("record_id")}
    if verdict == "BLOCK":
        corrections = response.get("corrections") or []
        if corrections and all(c.get("action") in KNOWN_ACTIONS
                                for c in corrections) and all(
                _correction_authorized(c, mandate)
                for c in corrections):
            return "fix", {"corrections": corrections,
                           "record_id": response.get("record_id"),
                           "max_attempts": response.get("max_attempts")}
        return "escalate", {"reason": "BLOCK without usable corrections",
                            "record_id": response.get("record_id")}
    # REVIEW, or anything unrecognized: stop and ask the buyer.
    # Never guess, never widen the authorization.
    return "escalate", {"reason": f"verdict {verdict} needs the buyer",
                        "record_id": response.get("record_id")}



## The flow, end to end (no model needed)

First with a scripted cart, so the routing is unmistakable: a wrong-variant cart is BLOCKed with an exact correction, the correction is applied, and the resubmission MATCHes.

In [ ]:
from datetime import datetime, timedelta, timezone

API_KEY = os.environ.get("PROVECART_API_KEY", "")  # the offline mock ignores it  # your integrator key

# --- The buyer's signed instructions, captured at order time --------
mandate = create_mandate(
    API_KEY,
    user_id="buyer-123",
    item_description="TrailVolt spare battery for a step-through frame e-bike",
    item_keywords=["trailvolt", "battery", "step-through"],
    category="ebike-parts",
    max_total_price=500.00,
    currency="USD",
    max_quantity=1,
    expires_at=(datetime.now(timezone.utc) + timedelta(minutes=30)).isoformat(),
)
print("mandate:", mandate["id"])

MERCHANT = {"name": "TrailVolt Store", "domain": "store.example.com"}

# --- Attempt 1: the agent built the WRONG variant -------------------
# The standard-frame battery does not fit a step-through frame; the
# fit fact lives in the product prose, which is exactly how these
# mistakes happen in the wild.
cart = [{"name": "TrailVolt Spare Battery", "variant": "standard frame",
         "quantity": 1, "unit_price": 431.00, "category": "ebike-parts",
         "brand": "TrailVolt"}]
res = verify_cart(API_KEY, mandate["id"], MERCHANT, cart, 431.00)
decision, detail = route_verdict(res, mandate)
print(res["verdict"], "->", decision, detail.get("corrections", ""))

# --- Apply ONLY the named, in-authorization correction --------------
assert decision == "fix"
for c in detail["corrections"]:
    if c["action"] == "change_variant":
        cart[0]["variant"] = "step-through frame"  # what the buyer asked for

# --- Attempt 2: resubmit, linked to the first check ------------------
res2 = verify_cart(API_KEY, mandate["id"], MERCHANT, cart, 431.00,
                   previous_record_id=detail["record_id"])
decision2, _ = route_verdict(res2, mandate)
print(res2["verdict"], "->", decision2)
assert decision2 == "proceed"
print("MATCH: payment may be captured now (and not a moment before).")


In [ ]:
# --- The other branch: when the instruction does not settle it -----
# A second buyer says only "a spare battery for my TrailVolt, under
# $500" -- no frame type. The agent must not guess the fit.
mandate2 = create_mandate(
    API_KEY,
    user_id="buyer-456",
    item_description="TrailVolt spare battery",
    item_keywords=["trailvolt", "battery"],
    category="ebike-parts",
    max_total_price=500.00,
    currency="USD",
    max_quantity=1,
    expires_at=(datetime.now(timezone.utc) + timedelta(minutes=30)).isoformat(),
)
res3 = verify_cart(API_KEY, mandate2["id"], MERCHANT, cart, 431.00)
decision3, detail3 = route_verdict(res3, mandate2)
print(res3["verdict"], "->", decision3, detail3.get("reason", ""))
# REVIEW routes to 'escalate': the flow stops here and asks the
# buyer which frame they have. No payment is captured on a guess.


In [ ]:
# --- Regression: a correction OUTSIDE the mandate must not fix ---
# The mandate above authorizes USD. A BLOCK whose correction would
# change the currency to EUR uses a KNOWN action, but its value is
# not in the buyer's authorization, so it must escalate, not fix.
rogue = {"verdict": "BLOCK", "record_id": "rec_rogue",
         "max_attempts": 3,
         "corrections": [{"action": "change_currency",
                          "field": "currency",
                          "expected": "EUR", "actual": "USD"}]}
decision_r, _ = route_verdict(rogue, mandate)
assert decision_r == "escalate", decision_r
in_mandate = {"verdict": "BLOCK", "record_id": "rec_ok",
              "max_attempts": 3,
              "corrections": [{"action": "change_currency",
                               "field": "currency",
                               "expected": "USD", "actual": "EUR"}]}
decision_ok, _ = route_verdict(in_mandate, mandate)
assert decision_ok == "fix", decision_ok
print("out-of-mandate correction -> escalate; in-mandate -> fix")

# --- Regression 2: routing follows mandate_id, not a global ----
# Mandate A authorizes USD, mandate B authorizes EUR. A BLOCK
# correction changing the currency to USD is inside A but
# outside B, so routed against B it must escalate, not fix.
# (The Agents SDK / Claude tool below routes through
# mandate_for_id(mandate_id) for exactly this reason.)
mandate_a = {**mandate, "id": "mandate_a", "currency": "USD"}
mandate_b = {**mandate, "id": "mandate_b", "currency": "EUR"}
MANDATES_BY_ID["mandate_a"] = mandate_a
MANDATES_BY_ID["mandate_b"] = mandate_b
usd_fix = {"verdict": "BLOCK", "record_id": "rec_usd",
           "max_attempts": 3,
           "corrections": [{"action": "change_currency",
                            "field": "currency",
                            "expected": "USD", "actual": "EUR"}]}
decision_b, _ = route_verdict(usd_fix, mandate_for_id("mandate_b"))
assert decision_b == "escalate", decision_b
decision_a, _ = route_verdict(usd_fix, mandate_for_id("mandate_a"))
assert decision_a == "fix", decision_a
assert mandate_for_id("mandate_unknown") is None
print("two-mandate routing: USD fix escalates for B, fixes for A")


## The same check as an Agents SDK tool

Wrapped as a function tool, the check becomes part of the agent's own loop: the agent builds the cart, calls the tool before paying, applies a `fix` decision's corrections itself, and stops on `escalate`.

In [ ]:
from agents import Agent, Runner, function_tool


@function_tool
def check_cart_before_payment(mandate_id: str, cart_json: str,
                              total_price: float,
                              previous_record_id: str = "") -> str:
    """Check a built cart against the buyer's signed instructions
    with ProveCart BEFORE paying. Returns proceed, fix (with the
    exact corrections to apply), or escalate (ask the buyer).
    On a re-check after applying a 'fix', pass the previous
    check's record_id as previous_record_id, so the resubmission
    stays linked to the check it corrects."""
    items = json.loads(cart_json)
    res = verify_cart(os.environ.get("PROVECART_API_KEY", ""), mandate_id,
                      MERCHANT, items, total_price,
                      previous_record_id=previous_record_id or None)
    signed = mandate_for_id(mandate_id)
    if signed is None:
        return json.dumps({
            "decision": "escalate",
            "reason": ("no signed mandate on record for this "
                       "mandate_id; refusing to route the check "
                       "against a different mandate")})
    decision, detail = route_verdict(res, signed)
    return json.dumps({"decision": decision, **detail})


agent = Agent(
    name="Careful Shopper",
    instructions=(
        "You buy exactly what the buyer's mandate authorizes. "
        "ALWAYS call check_cart_before_payment before paying. "
        "On 'fix', rebuild the cart applying only the named "
        "corrections and check again, passing the previous "
        "check's record_id as previous_record_id. On "
        "'escalate', stop and "
        "report what the buyer must decide. Pay only on 'proceed'."
    ),
    tools=[check_cart_before_payment],
)

# result = Runner.run_sync(agent, f"Buy the battery for mandate {mandate['id']}.")
# print(result.final_output)


## Make it a habit: get the agent certified

A check in the flow catches one order. Certification proves the
*habit*. ProveCart Verified Agent is a self-serve exam: the agent
works through graded buying situations — traps included — deciding
each one itself, including knowing when to ask instead of guess.

- **Practice first** (ungraded): twelve fixed instances.
- **Sign up** with no pre-issued key; the returned scoped key is
  shown once.
- **Take the live exam**: each session is generated fresh, and the
  shelf order is shuffled, so memorizing positions earns nothing.
- **Score 80 or above** and the agent version is certified for 90
  days, with a public result page and badge it can carry.

What certification is — and is not: it is a *published exam result*,
proof of how this agent actually behaved on measured decisions.
It gates nothing; stores and buyers remain free to choose any
agent. What it gives the agent is practical: the practice and the
exam teach the stop-fix-ask habit, so fewer of its purchases get
stopped mid-checkout and its buyers get a smoother ride — and its
builder gets a result they can point to instead of a claim.


In [ ]:
# ProveCart Verified Agent -- self-serve, no pre-issued key.
# Steps 1-2 call the live service, so they run only in LIVE_MODE;
# offline they are described, not executed.
if LIVE_MODE:
    # 1) Practice (ungraded, retry freely): twelve fixed instances.
    import urllib.request
    practice = json.loads(urllib.request.urlopen(
        "https://provecart.com/practice-set.json", timeout=20).read())
    print(f"practice instances: {len(practice)}")

    # 2) Sign up: returns a scoped api_key, shown once. Keep it.
    signup_status, signup_response = pc_post(
        "/agents/signup?src=cookbook",
        {"agent_name": "my-shopping-agent"}, api_key="")
    print("agent_id:", signup_response.get("agent_id"))
else:
    print("LIVE_MODE is False: practice fetch and signup skipped. "
          "Set LIVE_MODE = True to run them.")

# 3) Start the live exam (needs the key from step 2):
#    POST /agents/exam/start  {"agent_id": ...}   (X-API-Key header)
#    Each task shows a customer instruction and a shelf of options;
#    option order is shuffled and carries no signal.
# 4) Answer with your own decisions:
#    POST /agents/exam/answer -- decide from the shopper-facing
#    data; when an instruction genuinely does not settle the
#    choice, submit {"ask": true} instead of guessing.
# Score = 100 x (decisions that ended safely / tasks).
# 80 or above certifies for 90 days; renew with a fresh session.


## Notes

- The check is one API call (`POST /verify`) against a mandate;
  the full format is published as the Agent Correction Format
  (ACF v0.2) at https://provecart.com/standards/correction-format
- Verdicts are signed and every check is sealed to an evidence
  record, so a disputed order can be audited after the fact.
- A correction is only auto-applied when it brings the cart *into*
  the buyer's existing authorization. Anything needing more money,
  a different product, or different terms goes back to the buyer
  as a REVIEW.
